# P0 · Anomaly follow-up: words localize, coresets detect — why?

At matched memory (M stored vectors = K words), the codebook wins pixel AUROC in 15/15 MVTec categories,
while PatchCore's greedy coreset wins image AUROC. This notebook tests the explanation:

| Check | Prediction if the explanation is right |
|---|---|
| **AUPRO** (every defect region counts equally) | codebook still beats the coreset on localization |
| **Normal-pixel spread** | codebook background is quieter (lower relative IQR, higher pixel d′) |
| **Top-k image score** (mean of the k highest patches) | closes most of the codebook's image-AUROC gap |
| **Pill false alarms** | the codebook's worst normal images peak on speckles or print |
| **Hybrid** (coreset image score + codebook map) | best of both |

DINOv2-S only; 3 seeds. **Runtime (T4):** about 2 h, mostly k-means on CPU. Vocabularies are cached on
Drive, so a rerun after a disconnect resumes quickly. Uses the MVTec archive already on Drive.

## 1 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: DINOv2 extraction will be slow"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the code. Option A: GitHub (default). Option B: set REPO_URL = "" to install
# from bovw-forensics.zip uploaded to MyDrive/bovw-forensics/.
REPO_URL = "https://github.com/M3R2T2L2/bovw_forensics.git"
ZIP_PATH = "/content/drive/MyDrive/bovw-forensics/bovw-forensics.zip"

import os, shutil, subprocess, zipfile
shutil.rmtree("/content/bovw_forensics", ignore_errors=True)
if REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/bovw_forensics"], check=True)
else:
    zipfile.ZipFile(ZIP_PATH).extractall("/content/")
    os.rename("/content/bovw-forensics", "/content/bovw_forensics")

%cd /content/bovw_forensics
!pip install -q -e ".[dev]"   # torch, torchvision, transformers come preinstalled on Colab

In [ ]:
# 30-second check that the torch-free core works in this runtime
!python -m pytest -q

In [ ]:
# STL-10 on Drive: restore it here (skips the ~45 min download), or save it after the first download.
import os, shutil
DRIVE_STL = "/content/drive/MyDrive/bovw-forensics/data/stl10_binary"
LOCAL_STL = "/content/data/stl10_binary"
if os.path.isdir(DRIVE_STL) and not os.path.isdir(LOCAL_STL):
    shutil.copytree(DRIVE_STL, LOCAL_STL); print("Restored STL-10 from Drive")
elif os.path.isdir(LOCAL_STL) and not os.path.isdir(DRIVE_STL):
    shutil.copytree(LOCAL_STL, DRIVE_STL); print("Saved STL-10 to Drive for future runtimes")
else:
    print("STL-10 will download on first use; re-run this cell afterwards to save it to Drive"
          if not os.path.isdir(LOCAL_STL) else "STL-10 already local and on Drive")

## 2 · Data
The archive is kept on Drive so it downloads only once; each new runtime extracts it locally (~3–5 min).

In [ ]:
import os, subprocess
from bovw import ad_data
DRIVE_TAR = "/content/drive/MyDrive/bovw-forensics/data/mvtec_anomaly_detection.tar.xz"
LOCAL_ROOT = "/content/mvtec"

# Set to True only after accepting the licence on the MVTec page above.
I_ACCEPT_MVTEC_LICENCE = False

if not os.path.isfile(DRIVE_TAR) or os.path.getsize(DRIVE_TAR) < 1e9:   # missing, or a stub from a failed download
    if not I_ACCEPT_MVTEC_LICENCE:
        raise SystemExit("Accept the MVTec AD licence, then set I_ACCEPT_MVTEC_LICENCE = True "
                         "(or upload mvtec_anomaly_detection.tar.xz to MyDrive/bovw-forensics/data/).")
# Downloads only if missing or corrupt; verifies the SHA-256 (hashing ~5 GB takes about a minute).
ad_data.ensure_archive(DRIVE_TAR)

if not os.path.isdir(f"{LOCAL_ROOT}/bottle"):
    os.makedirs(LOCAL_ROOT, exist_ok=True)
    subprocess.run(["tar", "-xf", DRIVE_TAR, "-C", LOCAL_ROOT], check=True)
print(sorted(os.listdir(LOCAL_ROOT)))

## 3 · Run

In [ ]:
from bovw.sweep import load_config
from bovw import anomaly, plots

cfg = load_config("configs/p0_anomaly_mvtec_v2.yaml")
cfg["data"]["root"] = LOCAL_ROOT
df = anomaly.run(cfg)

## 4 · Results

`img` = max-patch image AUROC; `img_topK` = mean of the K highest patches; `pix` = pixel AUROC;
`aupro` = region overlap up to 30% FPR; `pixel_dprime` = defect-vs-normal separation in normal-pixel SDs.
Means over 15 categories and 3 seeds.

In [ ]:
t = anomaly.followup_table(df)
t.to_csv(f"{cfg['results_dir']}/{cfg['name']}_followup.csv")
t.round(3)

In [ ]:
# Per-category AUPRO at M = K = 1024 (codebook vs coreset)
d = df[df.k.fillna(0).astype(int).isin([0, 1024])]
d.pivot_table(index="category", columns="method", values="aupro").round(3)

## 5 · Pill: normal-pixel spread and false alarms

In [ ]:
from bovw import ad_data
pill = ad_data.load_mvtec(LOCAL_ROOT, "pill", cfg["data"]["size"], cfg["data"]["mask_size"])
spread = anomaly.normal_pixel_spread(cfg["results_dir"], cfg["name"], "pill", pill.test_masks)
spread.to_csv(f"{cfg['results_dir']}/{cfg['name']}_pill_spread.csv", index=False)
spread.round(3)

In [ ]:
fig = plots.false_alarms(cfg["results_dir"], cfg["name"], "pill", pill.test_images, pill.test_labels)
fig.savefig(f"{cfg['results_dir']}/{cfg['name']}_pill_false_alarms.png", dpi=110, bbox_inches="tight")